## 1. Ი𐑼 Instalación de Dependencias
Instalación de paquetes externos requeridos para la manipulación de documentos en Python.

In [ ]:
!pip install pypdf python-docx openpyxl pandas -q

## 2. Ი𐑼 Importacion de modulos
Importación de librerías nativas y externas para la lógica del procesamiento.

In [ ]:
#Lectura de documentos
import os
import pandas as pd
import pypdf
import docx
import pprint
from docx import Document

#Importación y conexión
import json
import os
from pathlib import Path


## 3. Ი𐑼 Conexión a Drive
Conexión con el entorno de Google Colab y montaje de Google Dirve

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Ი𐑼 Funciones de Extracción



*   leer_txt(): Extrae texto plano manejando codificación UTF-8.
*   leer_pdf(): Revisa y extrae el texto página por página usando pypdf.
*   leer_word(): Extrae párrafos de archivos .docx omitiendo líneas vacías.
*   leer_excel(): Recorre todas las pestañas de una hoja de cálculo y las convierte en texto estructurado.










*Ი𐑼 Lectura de archivos .txt*

In [ ]:
def leer_txt(ruta_archivo):
    """Abre y leer el contenido de archivos .TXT plano con codificación UTF-8."""
    try:
        with open(ruta_archivo, "r", encoding="utf-8", errors="ignore") as f:
            return f.read()
    except Exception as e:
        return f"[Error leyendo TXT: {str(e)}]"

*Ი𐑼 Lectura de archivos .pdf*

In [ ]:
def leer_pdf(ruta_archivo):
    """Extrae texto de archivos .PDF página por página."""
    texto = ""
    try:
        lector = pypdf.PdfReader(ruta_archivo)
        for pagina in lector.pages:
            try:
                contenido = pagina.extract_text()
                if contenido:
                    texto += contenido + "\n"
            except Exception:
                continue
    except Exception as e:
        return f"[Error leyendo PDF: {str(e)}]"
    return texto

*Ი𐑼 Lectura de archivos .docx*

In [ ]:
def leer_word(ruta_archivo):
    """Extrae texto de archivos .DOCX"""
    try:
        doc = docx.Document(ruta_archivo)
        return "\n".join([p.text for p in doc.paragraphs if p.text.strip()])
    except Exception as e:
        return f"[Error leyendo DOCX: {str(e)}]"

*Ი𐑼 Lectura de archivos .xlsx*

In [ ]:
def leer_excel(ruta_archivo):
    """Extrae y formatea el contenido de archivos .XLSX/XLS"""
    try:
        dict_hojas = pd.read_excel(ruta_archivo, sheet_name=None)
        texto_completo = []
        for nombre_hoja, df in dict_hojas.items():
            texto_completo.append(f"--- Hoja: {nombre_hoja} ---")
            texto_completo.append(df.to_string(index=False))
        return "\n".join(texto_completo)
    except Exception as e:
        return f"[Error leyendo Excel: {str(e)}]"

## 5. Ი𐑼 (pipeline_procesar_documento)



1.   Valida la existencia del archivo en el sistema.
2.   Identifica la extensión del documento.
1.   Ejecuta la función de extracción correspondiente.
2.   Normaliza y limpia los espacios en blanco sobrantes.
1.   Retorna un diccionario estructurado con los metadatos y el contenido.


*Ი𐑼 Pipeline*

In [ ]:
def pipeline_procesar_documento(ruta_archivo):
    """Identifica el tipo de archivo y ejecuta el lector correspondiente."""
    path = Path(ruta_archivo)
    ext = path.suffix.lower()

    if ext == ".txt":
        contenido = leer_txt(ruta_archivo)
    elif ext == ".pdf":
        contenido = leer_pdf(ruta_archivo)
    elif ext in [".docx", ".doc"]:
        contenido = leer_word(ruta_archivo)
    elif ext in [".xlsx", ".xls"]:
        contenido = leer_excel(ruta_archivo)
    else:
        return None  # Formato no soportado

    return {
        "nombre_archivo": path.name,
        "extension": ext,
        "contenido": contenido
    }

def procesar_carpeta_a_json(ruta_carpeta, ruta_salida_json="resultado_documentos.json"):
    """
    Recorre recursivamente una carpeta de Drive, procesa todos los documentos
    soportados utilizando el pipeline y guarda los resultados en un archivo JSON.
    """
    carpeta = Path(ruta_carpeta)
    extensiones_validas = {".pdf", ".docx", ".doc", ".txt", ".xlsx", ".xls"}
    resultados = []

    if not carpeta.exists():
        print(f"✘ Error: La ruta '{ruta_carpeta}' no existe.")
        return

    print(f"𓄲 Escaneando carpeta: {carpeta.name}...\n")

    for archivo in carpeta.rglob("*"):
        if archivo.is_file() and archivo.suffix.lower() in extensiones_validas:
            print(f"⌨ Procesando: {archivo.name}")
            try:
                datos = pipeline_procesar_documento(str(archivo))
                if datos:
                    datos["ruta"] = str(archivo)
                    resultados.append(datos)
            except Exception as e:
                resultados.append({
                    "nombre_archivo": archivo.name,
                    "ruta": str(archivo),
                    "error": str(e)
                })

    # Guardar archivo JSON
    with open(ruta_salida_json, "w", encoding="utf-8") as f:
        json.dump(resultados, f, ensure_ascii=False, indent=4)

    print(f"\n ✔ ¡Proceso completado con éxito!")
    print(f" 〽︎ Total de documentos procesados: {len(resultados)}")
    print(f" ⎙ Archivo generado: {ruta_salida_json}")

## 6. Ი𐑼 Prueba

In [ ]:
# Definir la ruta de la carpeta en Google Drive
RUTA_CARPETA_DRIVE = "/content/drive/MyDrive/0 Legislación SSO-20260724T184454Z-1-001/0 Legislación SSO"

# Ejecutar el procesamiento masivo
procesar_carpeta_a_json(RUTA_CARPETA_DRIVE)